<a href="https://colab.research.google.com/github/Raksh1707/taskdeeplearning/blob/main/task7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity

In [2]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3,
                                   stride=stride, padding=1, groups=in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x


In [3]:
class BottleneckBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = DepthwiseSeparableConv(out_channels, out_channels, stride)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.conv3 = nn.Conv2d(out_channels, out_channels * 4, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(out_channels * 4)

        self.relu = nn.ReLU()

        # Learnable Skip Projection
        self.skip = nn.Sequential(
            nn.Conv2d(in_channels, out_channels * 4, kernel_size=1, stride=stride),
            nn.BatchNorm2d(out_channels * 4)
        )

    def forward(self, x):
        identity = self.skip(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out += identity
        out = self.relu(out)
        return out


In [4]:
model = BottleneckBlock(64, 64)


In [5]:
x = torch.randn(1, 64, 56, 56)


In [6]:
output = model(x)
print("Output Shape:", output.shape)

Output Shape: torch.Size([1, 256, 56, 56])


In [7]:
params = sum(p.numel() for p in model.parameters())
print("Total Parameters:", params)

Total Parameters: 43520


In [8]:
with profile(
    activities=[ProfilerActivity.CPU],
    profile_memory=True,
    record_shapes=True
) as prof:
    model(x)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=5))


--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                    aten::conv2d         0.13%      43.956us        45.34%      15.251ms       3.050ms       8.42 MB           0 B             5  
               aten::convolution         0.43%     143.666us        45.21%      15.207ms       3.041ms       8.42 MB           0 B             5  
              aten::_convolution         0.26%      88.146us        44.78%      15.064ms       3.013ms       8.42 MB           0 B             5  
                aten::batch_norm         0.58%     194.506us        43.76%      14.721ms       3.680ms       7.66 MB  

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [9]:

# Channel Scaling
for c in [32, 64, 128]:
    model = BottleneckBlock(c, c)
    x = torch.randn(1, c, 56, 56)
    y = model(x)
    print(f"\nChannels: {c}")
    print("Output Shape:", y.shape)
    print("Parameters:", sum(p.numel() for p in model.parameters()))


Channels: 32
Output Shape: torch.Size([1, 128, 56, 56])
Parameters: 11520

Channels: 64
Output Shape: torch.Size([1, 256, 56, 56])
Parameters: 43520

Channels: 128
Output Shape: torch.Size([1, 512, 56, 56])
Parameters: 168960
